In [1]:
!pip install groq tavily-python python-dotenv


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### 1. Setup

In [2]:
import os
import re

from dotenv import load_dotenv
from groq import Groq

load_dotenv()

# Not GROQ_MODEL (openai/gpt-oss-120b): that model has native agentic tool-calling
# built in, and hijacks a text prompt describing an "Action:" step into a real
# tool-call attempt even with no tools bound. A plain instruct model just follows
# the ReAct text format we ask for, which is the point of building this by hand.
MODEL = "qwen/qwen3.6-27b"
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

client.chat.completions.create(
    model=MODEL, messages=[{"role": "user", "content": "Reply with just: ok"}]
).choices[0].message.content

'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:** The user says "Reply with just: ok"\n2.  **Identify Constraint:** The constraint is explicit: "Reply with just: ok"\n3.  **Determine Response:** I need to output exactly "ok" and nothing else.\n4.  **Verify Constraint:** Does the response contain only "ok"? Yes.\n5.  **Generate Output:** ok\n</think>\n\nok'

### 2. Tools

In [3]:
from tavily import TavilyClient

tavily = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))


def search(query: str) -> str:
    results = tavily.search(query, max_results=3)["results"]
    return "\n".join(f"- {r['title']}: {r['content'][:200]}" for r in results) or "No results."


_ALLOWED_NAMES = {"abs": abs, "round": round, "min": min, "max": max, "pow": pow}


def calculator(expression: str) -> str:
    try:
        value = eval(expression, {"__builtins__": {}}, _ALLOWED_NAMES)
        return str(value)
    except Exception as e:
        return f"Error: {e}"


TOOLS = {
    "search": (search, "Search the web for current facts, names, dates, news."),
    "calculator": (calculator, "Evaluate a Python arithmetic expression, e.g. '(3 + 4) * 2'."),
}

### 3. ReAct prompt

In [4]:
tool_descriptions = "\n".join(f"- {name}: {desc}" for name, (_, desc) in TOOLS.items())

SYSTEM_PROMPT = f"""Answer the question as best you can. You have access to these tools:

{tool_descriptions}

Rules:
- Never answer from memory, even if you are sure. Every fact must come from a
  search Observation and every computation must come from a calculator Observation.
- This applies even to a single simple arithmetic operation (e.g. one subtraction):
  never do arithmetic in a Thought, always call the calculator tool for it.
- Output ONLY one Thought, then either one Action+Action Input, or a Final Answer.
  Never write an Observation yourself, and never write more than one step at a time.
- Never send a Thought by itself: the Action+Action Input (or Final Answer) line
  must always come right after it, in the same response.

Use exactly this format:

Thought: reasoning about what to do next
Action: the tool to use, one of [{", ".join(TOOLS)}]
Action Input: the input to the tool
Observation: the tool's result (you do not write this, it is given to you)
... (this Thought/Action/Action Input/Observation cycle can repeat)
Thought: I now know the final answer
Final Answer: the final answer to the original question

Example:

Question: What is the population of the capital of France, divided by 1000?
Thought: I need to find the population of Paris first.
Action: search
Action Input: population of Paris
Observation: Paris has a population of about 2,102,000 (city proper).
Thought: Now I need to divide 2102000 by 1000.
Action: calculator
Action Input: 2102000 / 1000
Observation: 2102.0
Thought: I now know the final answer
Final Answer: About 2102 thousand people (2,102,000 / 1000).

Begin!"""

### 4. Agent loop

In [5]:
FINAL_RE = re.compile(r"Final Answer:\s*(.*)", re.DOTALL)
ACTION_RE = re.compile(r"Action:\s*(\w+)\s*\nAction Input:\s*(.*)", re.DOTALL)


def react_agent(question: str, max_steps: int = 6, verbose: bool = True) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Question: {question}"},
    ]

    for step in range(max_steps):
        response = client.chat.completions.create(
            model=MODEL, messages=messages, temperature=0.2, stop=["Observation:"]
        ).choices[0].message.content

        if verbose:
            print(f"--- step {step + 1} ---\n{response.strip()}\n")

        final = FINAL_RE.search(response)
        if final:
            return final.group(1).strip().strip("*")

        action = ACTION_RE.search(response)
        if not action:
            messages.append({"role": "assistant", "content": response})
            messages.append(
                {
                    "role": "user",
                    "content": (
                        "Observation: That response had no Action and no Final Answer. "
                        "Send a new Thought, immediately followed by either "
                        "'Action: <tool>\nAction Input: <input>' or 'Final Answer: <answer>'."
                    ),
                }
            )
            continue

        tool_name, tool_input = action.group(1).strip(), action.group(2).strip()
        tool_fn, _ = TOOLS.get(tool_name, (None, None))
        observation = tool_fn(tool_input) if tool_fn else f"Unknown tool '{tool_name}'."

        if verbose:
            print(f"Observation: {observation}\n")

        messages.append({"role": "assistant", "content": response})
        messages.append({"role": "user", "content": f"Observation: {observation}"})

    return "Agent stopped: max steps reached without a Final Answer."

### 5. Run it


In [6]:
answer = react_agent(
    "What year was the Bitcoin whitepaper published? "
    "Take that year, subtract 1969, and multiply the result by 3."
)
print("FINAL ANSWER:", answer)

--- step 1 ---
<think>
The user wants to know the year the Bitcoin whitepaper was published.
Then, I need to perform a calculation: (Year - 1969) * 3.

Step 1: Find the publication year of the Bitcoin whitepaper.
Step 2: Perform the calculation using the calculator tool.

I will start by searching for the publication year of the Bitcoin whitepaper.
</think>

Thought: I need to find the publication year of the Bitcoin whitepaper first.
Action: search
Action Input: Bitcoin whitepaper publication year



Observation: - The Bitcoin Whitepaper Explained: The Bitcoin whitepaper was written by a person or group using the pseudonym Satoshi Nakamoto. The paper was published on October 31, 2008. The true identity of Satoshi Nakamoto has never been confirme
- 10 years ago The Bitcoin Whitepaper was published by ...: Logo Logo

TwitterTwitter

Cryptocurrencies

# The Bitcoin whitepaper published by Satoshi Nakamoto

May 12, 2023 

Last updated:

The Bitcoin whitepaper published by Satoshi Nakamoto

Tiempo de lectu
- The Bitcoin White Paper Explained Simply - Chapter 1: Share article:



On October 31, 2008, a person calling himself by the alias Satoshi Nakamoto published a scientific paper entitled Bitcoin: A Peer-to-Peer Electronic Cash System. In this White Paper,



--- step 2 ---
<think>
The user wants to know the year the Bitcoin whitepaper was published.
I found the year is 2008.
The user wants me to take that year (2008), subtract 1969, and multiply the result by 3.
I need to perform the calculation: (2008 - 1969) * 3.
I must use the calculator tool for this.

Step 1: Confirm the year is 2008.
Step 2: Calculate (2008 - 1969) * 3.

I will now call the calculator.
</think>

Action: calculator
Action Input: (2008 - 1969) * 3

Observation: 117



--- step 3 ---
<think>
The user wants to know the result of a calculation based on the publication year of the Bitcoin whitepaper.
1.  **Find the publication year:** The search results indicate the Bitcoin whitepaper was published on October 31, 2008. So the year is 2008.
2.  **Perform the calculation:**
    *   Subtract 1969 from the year: $2008 - 1969$.
    *   Multiply the result by 3.
3.  **Use the calculator tool:** I have already called the calculator with `(2008 - 1969) * 3` and received the observation `117`.
4.  **Formulate the final answer:** The result is 117.

I have the final answer.
</think>

Final Answer: The Bitcoin whitepaper was published in 2008. Subtracting 1969 from 2008 gives 39. Multiplying 39 by 3 results in 117.

FINAL ANSWER: The Bitcoin whitepaper was published in 2008. Subtracting 1969 from 2008 gives 39. Multiplying 39 by 3 results in 117.
